# 04 KNN comparable bonds

Price a bond from the spreads of its k most similar bonds, as a trader does with comparables: how well it works and where it breaks.

Class notebooks followed: `2. KNN_IBM attrition` (the structure) and `1. Manual & scikit KNN` (the distance). Changes for our data: `KNeighborsRegressor` and MAE instead of the classifier and F1, time-ordered folds instead of `StratifiedKFold`, and searches on a 30% sample of the training months.

Validation months only: the test months are scored once, in `99_final`. Runs top to bottom in about 15 minutes on a laptop; figures go to
`reports/figures/04_*.png`.

In [ ]:
# Deck style for every figure: IG is always teal, HY always orange
import matplotlib.pyplot as plt

DECK = {"teal": "#114A6B", "orange": "#FF5F00", "slate": "#3A4550", "grey": "#8A949C", "pale": "#DCE6EC"}
plt.rcParams.update({
    "font.family": "Arial", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": DECK["grey"], "axes.labelcolor": DECK["slate"], "axes.titleweight": "bold",
    "axes.titlecolor": DECK["teal"],
    "axes.prop_cycle": plt.cycler(color=[DECK["teal"], DECK["orange"], DECK["grey"], DECK["slate"]]),
    "xtick.color": DECK["slate"], "ytick.color": DECK["slate"], "axes.grid": True, "axes.grid.axis": "y",
    "axes.axisbelow": True, "grid.color": DECK["pale"],
    "figure.figsize": (10, 5.6), "figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## 1. Setup

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline

from ml2.baseline import matrix_same_date
from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.evaluate import improvement_ci, metrics_by_stratum, regression_metrics
from ml2.features import make_preprocessor, xy
from ml2.splits import issuer_holdout_folds, walk_forward_folds

cfg = load_config()
TARGET = cfg["problem"]["target"]
PREFIX = "04"
FIG = resolve("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    """Save a figure to reports/figures/<prefix>_<name>.png."""
    fig.savefig(FIG / f"{PREFIX}_{name}.png", dpi=150, bbox_inches="tight")


check_panel()
df = load_panel(partitions=["train", "valid"])
train, valid = df[df.partition == "train"], df[df.partition == "valid"]
X_tr, y_tr = xy(train)
X_va, y_va = xy(valid)

# Searches run on a 30% sample of train (time order kept); chosen models are refitted on all of train
sub = train.sample(frac=0.3, random_state=42).sort_values("snapshot_date")
X_s, y_s = xy(sub)
cv = walk_forward_folds(sub.loc[X_s.index, "snapshot_date"])  # replaces cv=5 / KFold in the class notebooks
# New-issuer question: in each time-ordered fold, a fifth of the companies are held out (see ml2.splits)
new_issuer_folds = issuer_holdout_folds(sub.loc[X_s.index, "snapshot_date"], sub.loc[X_s.index, "issuer_ticker"])


def new_issuer_cost(model):
    """MAE on the held-out companies, trained with and without them (same number of rows, same months)."""
    seen, unseen = [], []
    for tr_seen, tr_unseen, held in new_issuer_folds:
        X_h, y_h = X_s.iloc[held], y_s.iloc[held]
        seen.append(mean_absolute_error(y_h, clone(model).fit(X_s.iloc[tr_seen], y_s.iloc[tr_seen]).predict(X_h)))
        unseen.append(mean_absolute_error(
            y_h, clone(model).fit(X_s.iloc[tr_unseen], y_s.iloc[tr_unseen]).predict(X_h)))
    return float(np.mean(seen)), float(np.mean(unseen))
print(f"train {X_tr.shape}, valid {X_va.shape}, search sample {X_s.shape}, {len(cv)} time-ordered folds")

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingRegressor
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CV = cv  # the class notebook's name

In [ ]:
# The bar: matrix pricing re-marked on each date (the gate baseline), on valid. About 1.5 minutes.
valid = valid.copy()
valid["pred_matrix_same_date"] = matrix_same_date(valid)
results = {}  # model name -> metrics by grade on valid, for the comparison table


def vs_bar(pred, name):
    """MAE by grade for predictions on X_va's rows, stored in `results`, shown next to the gate baseline."""
    rows = valid.loc[X_va.index].assign(pred=np.asarray(pred))
    results[name] = metrics_by_stratum(rows, TARGET, "pred", "grade_bucket").set_index("stratum")
    bar = metrics_by_stratum(rows, TARGET, "pred_matrix_same_date", "grade_bucket").set_index("stratum")
    out = pd.DataFrame({name: results[name]["mae_bps"], "matrix re-marked (bar)": bar["mae_bps"]}).T
    out["IG gain"] = 1 - out["IG"] / out.loc["matrix re-marked (bar)", "IG"]
    out["HY gain"] = 1 - out["HY"] / out.loc["matrix re-marked (bar)", "HY"]
    return out[["IG", "HY", "ALL", "IG gain", "HY gain"]].round(3)


vs_bar(valid.loc[X_va.index, "pred_matrix_same_date"], "Matrix pricing re-marked each date (bar)")

## 2. Standardization inside the pipeline

KNN_IBM. KNN measures distance between bonds, so every feature must be on the same scale: unscaled, `total_debt`
in dollars would swamp everything else. The scaler sits inside the pipeline so it is fitted on the training rows
only (fitting it on all rows would leak validation information).

In [ ]:
def make_knn(k=5, weights="distance"):
    # standardization happens inside the pipeline, fitted on the training rows only
    return Pipeline([("pre", make_preprocessor()), ("knn", KNeighborsRegressor(n_neighbors=k, weights=weights))])


make_knn()

## 3. KNN model 1

k = 5, closer neighbours weigh more (`weights="distance"`), fitted on all of train.

In [ ]:
knn1 = make_knn(5, "distance").fit(X_tr, y_tr)
vs_bar(knn1.predict(X_va), "KNN model 1 (k=5, distance)")

## 4. Look for the best k manually

KNN_IBM tries every k and plots the CV error. Here k from 2 to 40 on the time-ordered folds.

In [ ]:
k_values = list(range(2, 41, 2))
mae = [-cross_val_score(make_knn(k), X_s, y_s, cv=CV, scoring="neg_mean_absolute_error", n_jobs=-1).mean()
       for k in k_values]
best_k = k_values[int(np.argmin(mae))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, mae, marker="o"); ax.axvline(best_k, color="red", linestyle="--", label=f"best k = {best_k}")
ax.set_xlabel("Number of neighbors (k)"); ax.set_ylabel("Mean CV MAE (bp)"); ax.set_title("Error against k")
ax.legend()
save(fig, "error_vs_k")

**Reading** (to write from the output above). Explain the U shape: k = 1 memorizes (low bias, high variance), a large k averages everything (high bias).

## 5. KNN model 2: grid search

KNN_IBM: grid over k and the weighting.

In [ ]:
grid = GridSearchCV(make_knn(), {"knn__n_neighbors": k_values, "knn__weights": ["uniform", "distance"]},
                    scoring="neg_mean_absolute_error", cv=CV, n_jobs=-1).fit(X_s, y_s)
print(grid.best_params_, f"CV MAE {-grid.best_score_:.1f} bp")
k_best, w_best = grid.best_params_["knn__n_neighbors"], grid.best_params_["knn__weights"]
res = pd.DataFrame(grid.cv_results_)
print(res.groupby("param_knn__weights")["mean_test_score"].max().mul(-1).round(1))
knn_final = grid.best_estimator_.fit(X_tr, y_tr)
vs_bar(knn_final.predict(X_va), "KNN model 2 (grid search)")

**Reading** (to write from the output above). Which weighting wins, and why? (With distance weights a very close comparable counts more.)

## 6. KNN model 3: bagging of KNNs

KNN_IBM: preprocessing first, then a bag of KNNs, each on half the training rows.

In [ ]:
bagged = Pipeline([("pre", make_preprocessor()),
                   ("bag", BaggingRegressor(KNeighborsRegressor(n_neighbors=k_best, weights=w_best),
                                            n_estimators=10, max_samples=0.5, random_state=42, n_jobs=-1))])
bagged.fit(X_tr, y_tr)
vs_bar(bagged.predict(X_va), "KNN model 3 (bagging of KNNs)")

**Reading** (to write from the output above). Bagging helps unstable learners such as deep trees, whose predictions change a lot with the sample. KNN is stable: averaging several KNNs barely changes anything.

## 7. What makes bonds comparable?

KNN suffers when there are many features (the curse of dimensionality: in many dimensions every bond is far from
every other). A trader's short list: maturity, coupon, grade, sector, stock volatility, leverage, company size.

In [ ]:
short = ["years_to_maturity", "coupon_pct", "vol_252d", "leverage", "log_market_cap"]
pre_short = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), short),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["grade_bucket", "sector"]),
])
knn_short = Pipeline([("pre", pre_short), ("knn", KNeighborsRegressor(n_neighbors=k_best, weights=w_best))])
cv_all = -cross_val_score(make_knn(k_best, w_best), X_s, y_s, cv=CV, scoring="neg_mean_absolute_error",
                          n_jobs=-1).mean()
cv_short = -cross_val_score(knn_short, X_s, y_s, cv=CV, scoring="neg_mean_absolute_error", n_jobs=-1).mean()
print(f"CV MAE, all {X_s.shape[1]} features: {cv_all:.1f} bp; trader's short list: {cv_short:.1f} bp")
knn_short.fit(X_tr, y_tr)
vs_bar(knn_short.predict(X_va), "KNN, trader's short list")

## 8. Speed

KNN does no work when it trains (it stores the rows) and all of it when it predicts.

In [ ]:
t0 = time.perf_counter(); make_knn(k_best, w_best).fit(X_tr, y_tr); t_fit = time.perf_counter() - t0
t0 = time.perf_counter(); knn_final.predict(X_va); t_pred = time.perf_counter() - t0
print(f"fit {t_fit:.1f} s; predict {len(X_va):,} bonds {t_pred:.1f} s ({1000 * t_pred / len(X_va):.2f} ms per bond)")

**Reading** (to write from the output above). What does that mean for pricing tens of thousands of bonds every day?

## 9. New-issuer question

For a known company the nearest comparables are usually its own other bonds. In each time-ordered fold, a fifth
of the companies are held out and the model is trained twice on the same number of rows from the same months, with
and without those companies' earlier bonds; both are scored on the held-out companies only.

In [ ]:
seen_mae, unseen_mae = new_issuer_cost(grid.best_estimator_)
print(f"Held-out companies: {seen_mae:.1f} bp when their own bonds are comparables, {unseen_mae:.1f} bp when not")

**Reading** (to write from the output above). How much worse is it without the company's own bonds?

## 10. Show the comparables

For 3 bonds in valid, their 5 nearest training bonds under the trader's short list.

In [ ]:
show = ["issuer_name", "snapshot_date", "years_to_maturity", "grade_bucket", "sector", TARGET]
train_rows = train.loc[X_tr.index]
picks = X_va.sample(3, random_state=7).index
for i in picks:
    dist, idx = knn_short["knn"].kneighbors(knn_short["pre"].transform(X_va.loc[[i]]), n_neighbors=5)
    print("VALID BOND:"); print(valid.loc[[i], show].to_string(index=False))
    print(f"prediction {knn_short.predict(X_va.loc[[i]])[0]:.0f} bp; its 5 comparables:")
    print(train_rows.iloc[idx[0]][show].assign(distance=dist[0].round(2)).to_string(index=False)); print()

**Reading** (to write from the output above). Do the model's comparables look like a trader's? Are they mostly the same company's other bonds?

## 11. Comparison table and RUN_LOG

In [ ]:
table = pd.concat({name: r["mae_bps"] for name, r in results.items()}, axis=1).T[["IG", "HY", "ALL"]]
bar = table.loc["Matrix pricing re-marked each date (bar)"]
table["IG gain vs bar"] = 1 - table["IG"] / bar["IG"]
table["HY gain vs bar"] = 1 - table["HY"] / bar["HY"]
extra = pd.DataFrame({name: r.loc["ALL", ["rmse_bps", "r2"]] for name, r in results.items()}).T
table = table.join(extra).round(3)
table.to_csv(FIG.parent / f"{PREFIX}_comparison_valid.csv")
table

### Lines for RUN_LOG.md

For `notebooks/RUN_LOG.md`; the parameters are printed in the last cell.

In [ ]:
today = pd.Timestamp.today().strftime("%Y-%m-%d")
for name, r in table.iterrows():
    print(f"| {today} | <your name> | 04_knn_comparables | {name} | all 33 | fit on train, scored on valid | "
          f"{r['IG']:.1f} / {r['HY']:.1f} | gain vs bar IG {r['IG gain vs bar']:.1%}, "
          f"HY {r['HY gain vs bar']:.1%} |")

In [ ]:
print("best k and weights:", grid.best_params_)